In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")
print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

amx_path = os.path.join(path, 'Q3_data.csv')
df_amx = pd.read_csv(amx_path)

In [ ]:
# Task 2: Write your code here:
df_amx.head()

In [ ]:
# Task 3: Write your code here:
df_amx.info()

In [ ]:
# Task 4: Write your code here:
df_amx.describe()

In [ ]:
# Task 1: Write your code here:

df_clean = df_amx.copy()
features = df_amx.copy().drop(columns='Target')

# Analyze missing values
missing_percentage = (df_amx.isnull().sum() / len(df_amx)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

# The missing values in many col are greater than 95 > so we need to drop them




In [ ]:
# Task 2: Write your code here:

#Check and remove duplicates if any exist
print("Checking for duplicate rows...")
duplicate_rows = features.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")


In [ ]:
# Task 3: Write your code here:

# We don't have any catgorical and the target is already label encoded as binary

In [ ]:
# Task 4: Write your code here:

#Apply feature scaling for all features (Use StandardScaler)
feature_cols = features.columns
scaler = StandardScaler()
df_clean_scaled = scaler.fit_transform(df_clean[feature_cols])

print(f"\nScaled ranges - Min: {df_clean_scaled.min():.2f}, Max: {df_clean_scaled.max():.2f}")

In [ ]:
# Task 5: Write your code here:

# Plot the target distribution (Target)
plt.figure(figsize=(10, 5))
plt.hist(df_clean['Target'].dropna(), bins=50, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

# it is

In [ ]:
# Task 1: Write your code here:
#1. Split the dataset into features (X) and target (y)

# Define features (X) and target (y)
X = df_clean_scaled.copy()
y = df_clean['Target'].copy()

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier

models = {
"CatBoost": CatBoostClassifier(
n_estimators=200, # Number of boosting rounds
max_depth=4, # Maximum tree depth
verbose=0 # Suppress output
)
}


kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = {name: {'mse': [], 'mae': [], 'rmse': [], 'r2': []} for name in models}

for train_idx, test_idx in kf.split(X):
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

  for model_name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    # Predict
    y_pred = model.predict(X_test)
    # Evaluate
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    # Store results
    results[model_name]['mse'].append(mse)
    results[model_name]['mae'].append(mae)
    results[model_name]['rmse'].append(rmse)
    results[model_name]['r2'].append(r2)


# Print average results
for model_name in results:
  print(f"\n{model_name}:")
  print(f" MSE: {np.mean(results[model_name]['mse']):.4f}")
  print(f" MAE: {np.mean(results[model_name]['mae']):.4f}")
  print(f" RMSE: {np.mean(results[model_name]['rmse']):.4f}")
  print(f" R2: {np.mean(results[model_name]['r2']):.4f}")

In [ ]:
# Task 1: Write your code here:

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:

print("")

In [ ]:
# Task Bonus: Write your code here: